# Gold Layer - Hithub Repositories
## Process:
### 1. Import Libraries needed
### 2. Add Configuration
### - Silvers & Gold Paths
### 3. Load Silver Parquet File dynamically
### 4. Aggregate Sections
### - Top Languages by repository count (value_counts)
### - Top Users by repository count (groupby owner_login, sort_index = ascending)
### - Average stars by language
### 5. Convert Series to DF
### 6. Save to Gold Data Path

In [21]:
import pandas as pd
import os
from datetime import datetime
import pyarrow

In [22]:
SILVER_FOLDER = "C:/Users/rjaya/OneDrive/Desktop/hop_dataeng/data/silver"
GOLD_FOLDER = "C:/Users/rjaya/OneDrive/Desktop/hop_dataeng/data/gold"
COLLECTION_DATE = datetime.now().strftime("%Y-%m-%d")

In [23]:
silver_parquet_file = sorted([filename for filename in os.listdir(SILVER_FOLDER) if filename.endswith('.parquet')])
load_silver_file_path = f"{SILVER_FOLDER}/{silver_parquet_file[-1]}"

In [24]:
df = pd.read_parquet(load_silver_file_path)

print(f"Loaded: {load_silver_file_path}")
print(f"Records: {len(df)}")
print(f"Columns: {df.columns.tolist()}")

Loaded: C:/Users/rjaya/OneDrive/Desktop/hop_dataeng/data/silver/2025-11-19_github_users.parquet
Records: 1270
Columns: ['owner_login', 'name', 'description', 'language', 'stargazers_count', 'created_at', 'updated_at']


In [25]:
# 1 - Languages by repository count
language_repository_count = df['language'].value_counts()
print(f"Top language used: {language_repository_count}")

# 2 - Users by repository count
top_users_by_repository = df['owner_login'].value_counts().sort_index(key=lambda x: x.str.lower())
print(f"Top Users By Repository: {top_users_by_repository}")

# 3 - Average stars by language
average_stars_by_language = df.groupby('language')['stargazers_count'].mean().sort_values(ascending=False)
print(f"Average stars by language: {average_stars_by_language}")

Top language used: language
JavaScript                 335
Unknown                    331
Ruby                        67
Python                      56
HTML                        54
Go                          52
CSS                         50
PHP                         48
TypeScript                  42
C                           35
C#                          33
Java                        22
Shell                       19
C++                         16
Rust                        12
Kotlin                       9
Jupyter Notebook             9
R                            8
Elixir                       8
Objective-C                  8
CoffeeScript                 7
Vue                          6
Erlang                       5
Emacs Lisp                   3
SCSS                         2
Swift                        2
OpenSCAD                     2
Assembly                     2
HCL                          2
Tcl                          2
Verilog                      1
MDX        

In [26]:
# Convert Series to Dataframe
df_languages = language_repository_count.reset_index()
df_languages.columns = ['language', 'count']

df_users = top_users_by_repository.reset_index()
df_users.columns = ['user', 'count']

df_avg_stars = average_stars_by_language.reset_index()
df_avg_stars.columns = ['language', 'average_stars']

print(df_languages.head())
print(df_users.head())
print(df_languages.head())

     language  count
0  JavaScript    335
1     Unknown    331
2        Ruby     67
3      Python     56
4        HTML     54
         user  count
0  adamwathan     30
1  addyosmani     30
2    AndrewNg     20
3     antirez     30
4    bradfitz     30
     language  count
0  JavaScript    335
1     Unknown    331
2        Ruby     67
3      Python     56
4        HTML     54


In [27]:
# Verify if they look correct
print(f"Languages Dataframe: {df_languages}")
print(f"Users Dataframe: {df_users}")
print(f"AVG Stars Dataframe: {df_avg_stars}")

Languages Dataframe:                    language  count
0                JavaScript    335
1                   Unknown    331
2                      Ruby     67
3                    Python     56
4                      HTML     54
5                        Go     52
6                       CSS     50
7                       PHP     48
8                TypeScript     42
9                         C     35
10                       C#     33
11                     Java     22
12                    Shell     19
13                      C++     16
14                     Rust     12
15                   Kotlin      9
16         Jupyter Notebook      9
17                        R      8
18                   Elixir      8
19              Objective-C      8
20             CoffeeScript      7
21                      Vue      6
22                   Erlang      5
23               Emacs Lisp      3
24                     SCSS      2
25                    Swift      2
26                 OpenSCAD      2

In [28]:
# Create a File Path
os.makedirs(f"{GOLD_FOLDER}/parquet", exist_ok=True)
os.makedirs(f"{GOLD_FOLDER}/csv", exist_ok=True)

# Save files to Gold Folder
aggregations = {
    'top_languages': df_languages,
    'top_users': df_users,
    'avg_stars': df_avg_stars
}

for name, df in aggregations.items():
    parquet_path = f"{GOLD_FOLDER}/parquet/{COLLECTION_DATE}_{name}.parquet"
    csv_path = f"{GOLD_FOLDER}/csv/{COLLECTION_DATE}_{name}.csv"

    df.to_parquet(parquet_path, index=False)
    df.to_csv(csv_path, index=False)

    print(f"Saved {name}")



Saved top_languages
Saved top_users
Saved avg_stars
